In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.ops.supplier_truth AS
SELECT supplier_id, reliability_tier
FROM read_files('/Volumes/supplychainiq/ops/ground_truth/suppliers.csv', format => 'csv', header => true);

In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.gold.supplier_scorecard AS
WITH deliveries AS (                               -- one row per received PO
  SELECT p.supplier_id, p.po_id, p.qty_ordered, s.qty_shipped,
         datediff(s.arrival_date, p.expected_date) AS delay_days
  FROM supplychainiq.silver.purchase_orders p
  JOIN supplychainiq.silver.shipments s USING (po_id)
  WHERE p.status = 'received'),
perf AS (
  SELECT supplier_id,
         COUNT(*)                                                             AS pos_delivered,
         ROUND(AVG(CASE WHEN delay_days <= 0 THEN 1 ELSE 0 END), 3)           AS on_time_rate,
         ROUND(AVG(CASE WHEN qty_shipped >= qty_ordered THEN 1 ELSE 0 END), 3) AS in_full_rate,
         ROUND(AVG(CASE WHEN delay_days <= 0 AND qty_shipped >= qty_ordered
                        THEN 1 ELSE 0 END), 3)                                AS otif_rate,
         ROUND(AVG(GREATEST(delay_days, 0)), 2)                               AS avg_delay_days,
         ROUND(SUM(qty_shipped) / SUM(qty_ordered), 3)                        AS unit_fill_rate
  FROM deliveries GROUP BY supplier_id),
lost AS (                                          -- as-of join: which supplier supplied the item ON THAT DAY
  SELECT ps.supplier_id,
         SUM(f.units_sold)       AS units_sold,
         SUM(f.est_lost_units)   AS est_lost_units,
         SUM(f.est_lost_revenue) AS est_lost_revenue
  FROM supplychainiq.gold.fact_item_day f
  JOIN supplychainiq.silver.products_scd2 ps
    ON ps.item_id = f.item_id
   AND f.snapshot_date >= ps.valid_from
   AND (ps.valid_to IS NULL OR f.snapshot_date < ps.valid_to)
  WHERE f.snapshot_date >= '2014-06-20'
  GROUP BY ps.supplier_id)
SELECT p.*,
       ROUND(l.est_lost_revenue, 0)                                          AS est_lost_revenue,
       ROUND(l.est_lost_units / (l.units_sold + l.est_lost_units), 4)        AS lost_sales_rate
FROM perf p LEFT JOIN lost l USING (supplier_id);

In [0]:
%sql
SELECT s.supplier_id, t.reliability_tier, s.pos_delivered, s.on_time_rate, s.in_full_rate,
       s.otif_rate, s.avg_delay_days, s.lost_sales_rate, s.est_lost_revenue
FROM supplychainiq.gold.supplier_scorecard s
JOIN supplychainiq.ops.supplier_truth t USING (supplier_id)
ORDER BY s.otif_rate;

In [0]:
%sql
SELECT t.reliability_tier, COUNT(*) AS suppliers,
       MIN(s.otif_rate) AS min_otif, MAX(s.otif_rate) AS max_otif,
       ROUND(AVG(s.lost_sales_rate), 4) AS avg_lost_rate,
       ROUND(SUM(s.est_lost_revenue)) AS est_lost_revenue
FROM supplychainiq.gold.supplier_scorecard s
JOIN supplychainiq.ops.supplier_truth t USING (supplier_id)
GROUP BY 1;